# 3D COM Kyphosis Calculator

A thoracic kyphosis calculator for users who do not write code. Open it and press **Runtime → Run all**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Novestars/3d_com_kyphosis/blob/main/3D_COM_kyphosis_calculator.ipynb)

**What this is.** It takes whole-volume vertebral-body centers of mass (COMs), draws a T1–T2 chord and a T8–L2 chord, and reports the acute angle between them. This is a centerline surrogate, not a human endplate Cobb angle.

**Do not select a GPU.** Leave the runtime on CPU.

## How to use this notebook

1. Click **Open in Colab** above, or upload this `.ipynb` to Colab.
2. Menu **Runtime → Run all**. Block 1 installs the packages.
3. When you reach Upload: choose a Google Drive folder, or upload a zip / `.nii.gz` files.
4. Check the printed case count and the label list from the first case.
5. If the labels do not match: change **only Block 3**, then run from Block 3 downward.
6. The last block writes and downloads `3D_COM_kyphosis_angles.xlsx`.

**If the label indices change, edit only Block 3.** Leave the other collapsed blocks closed. Click the triangle next to a heading if you need to see the code.



## 1. Install packages



In [ ]:
#@title 1. Install packages { display-mode: "form" }
import importlib.util
import subprocess
import sys

PACKAGES = ["nibabel", "pandas", "openpyxl", "scipy", "matplotlib"]
missing = [name for name in PACKAGES if importlib.util.find_spec(name) is None]
if missing:
    print("Installing:", ", ".join(missing))
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    except subprocess.CalledProcessError as error:
        raise SystemExit(
            "Install failed. Check the network and re-run this block. "
            f"Original error: {error}"
        ) from error
    print("Install finished.")
else:
    print("Packages ready: nibabel, pandas, openpyxl, scipy, matplotlib")


## 2. Algorithm



In [ ]:
#@title 2. Load the algorithm (do not edit) { display-mode: "form" }
from __future__ import annotations

import re
import zipfile
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import find_objects


def parse_level_list(text: str) -> tuple[str, ...]:
    levels = tuple(part.strip().upper() for part in str(text).split(",") if part.strip())
    if len(levels) < 2:
        raise ValueError("Upper and lower lines each need at least two levels, for example T1,T2")
    return levels


def chord_direction(first: np.ndarray, second: np.ndarray) -> tuple[np.ndarray, float]:
    direction = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
    length = float(np.linalg.norm(direction))
    if not np.isfinite(length) or length <= np.finfo(float).eps:
        raise ValueError("degenerate COM chord")
    direction /= length
    if direction[1] < 0:
        direction *= -1.0
    return direction, length


def angle_between_lines(first: np.ndarray, second: np.ndarray) -> float:
    cosine = abs(float(np.dot(first, second))) / (
        float(np.linalg.norm(first)) * float(np.linalg.norm(second))
    )
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def signed_angle(first: np.ndarray, second: np.ndarray) -> float:
    determinant = float(first[0] * second[1] - first[1] * second[0])
    return float(np.degrees(np.arctan2(determinant, float(np.dot(first, second)))))


def physical_centroids(
    data: np.ndarray,
    affine: np.ndarray,
    level_to_label: dict[str, int],
    required_levels: tuple[str, ...],
) -> tuple[dict[str, np.ndarray], list[dict], list[str]]:
    labels = [level_to_label[level] for level in required_levels]
    boxes = find_objects(data, max_label=max(labels))
    points: dict[str, np.ndarray] = {}
    rows: list[dict] = []
    missing: list[str] = []
    for level in required_levels:
        label = int(level_to_label[level])
        box = boxes[label - 1] if label - 1 < len(boxes) else None
        if box is None:
            missing.append(level)
            continue
        local = np.argwhere(np.asarray(data[box]) == label)
        if not len(local):
            missing.append(level)
            continue
        offset = np.array([axis.start for axis in box], dtype=float)
        mean_index = local.mean(axis=0) + offset
        world = np.asarray(affine, dtype=float) @ np.r_[mean_index, 1.0]
        points[level] = np.asarray(world[:3], dtype=float)
        rows.append(
            {
                "level": level,
                "label_id": label,
                "voxel_count": int(len(local)),
                "mean_i": float(mean_index[0]),
                "mean_j": float(mean_index[1]),
                "mean_k": float(mean_index[2]),
                "lr_mm": float(world[0]),
                "ap_mm": float(world[1]),
                "si_mm": float(world[2]),
            }
        )
    return points, rows, missing


def case_id_from_path(path: Path) -> str:
    if path.name.startswith("seg.") and path.parent.name:
        parent = path.parent.name
        return str(int(parent)) if parent.isdigit() else parent
    stem = path.name
    if stem.endswith(".nii.gz"):
        stem = stem[:-7]
    elif stem.endswith(".nii"):
        stem = stem[:-4]
    match = re.search(r"(\d+)", stem)
    return match.group(1) if match else stem


def discover_cases(root: Path) -> list[Path]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Folder not found: {root}")
    segs = sorted(path for path in root.rglob("seg.nii.gz") if path.is_file())
    if segs:
        return segs
    files = []
    for path in root.rglob("*"):
        name = path.name.lower()
        if path.is_file() and (name.endswith(".nii.gz") or name.endswith(".nii")):
            files.append(path)
    return sorted(files)


def measure_case(
    seg_path: Path,
    level_to_label: dict[str, int],
    upper_levels: tuple[str, ...],
    lower_levels: tuple[str, ...],
) -> tuple[dict, list[dict]]:
    required_levels = tuple(dict.fromkeys([*upper_levels, *lower_levels]))
    case_id = case_id_from_path(Path(seg_path))
    base = {
        "case_id": case_id,
        "segmentation_path": str(seg_path),
        "angle_deg": np.nan,
        "signed_angle_deg": np.nan,
        "upper_chord_length_mm": np.nan,
        "lower_chord_length_mm": np.nan,
        "anatomic_order_ok": False,
        "qc_pass": False,
        "qc_reason": "",
        "missing_labels": "",
        "orientation": "",
        "spacing_lr_mm": np.nan,
        "spacing_ap_mm": np.nan,
        "spacing_si_mm": np.nan,
    }
    try:
        image = nib.load(str(seg_path))
        data = np.asanyarray(image.dataobj)
        if data.ndim != 3:
            raise ValueError(f"expected 3D segmentation, got shape {data.shape}")
        points, point_rows, missing = physical_centroids(
            data, image.affine, level_to_label, required_levels
        )
        if missing:
            raise ValueError("missing required labels: " + ",".join(missing))
        upper_direction, upper_length = chord_direction(
            points[upper_levels[0]][[1, 2]],
            points[upper_levels[-1]][[1, 2]],
        )
        lower_direction, lower_length = chord_direction(
            points[lower_levels[0]][[1, 2]],
            points[lower_levels[-1]][[1, 2]],
        )
        angle = angle_between_lines(upper_direction, lower_direction)
        signed = signed_angle(upper_direction, lower_direction)
        si_sequence = np.array([points[level][2] for level in required_levels])
        order_ok = bool(np.all(np.diff(si_sequence) < 0))
        zooms = tuple(float(value) for value in image.header.get_zooms()[:3])
        base.update(
            {
                "angle_deg": angle,
                "signed_angle_deg": signed,
                "upper_chord_length_mm": upper_length,
                "lower_chord_length_mm": lower_length,
                "anatomic_order_ok": order_ok,
                "qc_pass": bool(np.isfinite(angle) and 0.0 <= angle <= 90.0),
                "qc_reason": "" if order_ok else "non-monotonic SI order; angle retained",
                "orientation": "".join(nib.aff2axcodes(image.affine)),
                "spacing_lr_mm": zooms[0],
                "spacing_ap_mm": zooms[1],
                "spacing_si_mm": zooms[2],
            }
        )
        for row in point_rows:
            row["case_id"] = case_id
        return base, point_rows
    except Exception as error:
        missing_text = ""
        message = str(error)
        if "missing required labels" in message:
            missing_text = message.split(":", 1)[-1].strip()
        base["qc_reason"] = f"{type(error).__name__}: {error}"
        base["missing_labels"] = missing_text
        return base, []


def unique_labels(seg_path: Path, max_preview: int = 40) -> list[int]:
    image = nib.load(str(seg_path))
    values = np.unique(np.asanyarray(image.dataobj))
    labels = [int(value) for value in values if int(value) != 0]
    return labels[:max_preview]


def write_excel(
    cases: pd.DataFrame,
    points: pd.DataFrame,
    config: dict,
    out_path: Path,
) -> Path:
    out_path = Path(out_path)
    valid = cases.loc[cases["qc_pass"].fillna(False) & cases["angle_deg"].notna()]
    summary = pd.DataFrame(
        [
            {"item": "cases_found", "value": int(len(cases))},
            {"item": "qc_pass", "value": int(cases["qc_pass"].fillna(False).sum())},
            {"item": "qc_fail", "value": int((~cases["qc_pass"].fillna(False)).sum())},
            {
                "item": "angle_mean_deg",
                "value": float(valid["angle_deg"].mean()) if len(valid) else np.nan,
            },
            {
                "item": "angle_min_deg",
                "value": float(valid["angle_deg"].min()) if len(valid) else np.nan,
            },
            {
                "item": "angle_max_deg",
                "value": float(valid["angle_deg"].max()) if len(valid) else np.nan,
            },
            {"item": "upper_line", "value": "-".join(config["upper_levels"])},
            {"item": "lower_line", "value": "-".join(config["lower_levels"])},
            {
                "item": "label_map",
                "value": ", ".join(
                    f"{level}={label}" for level, label in config["level_to_label"].items()
                ),
            },
        ]
    )
    readme = pd.DataFrame(
        [
            ["3D COM kyphosis", "Whole-volume vertebral-body COM chord angle"],
            ["Method", "3D COM -> keep AP-SI -> upper chord -> lower chord -> acute angle"],
            ["Upper line", "First and last listed upper vertebrae, default T1-T2"],
            ["Lower line", "First and last listed lower vertebrae, default T8-L2"],
            ["If labels change", "Edit only notebook block 3 (the label form). Do not edit other cells."],
            ["QC fail", "Missing required label, empty mask, not 3D, or degenerate chord"],
            ["Not Cobb", "Centerline surrogate, not radiographic endplate Cobb"],
        ],
        columns=["item", "definition"],
    )
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        cases.to_excel(writer, sheet_name="Per Case", index=False)
        points.to_excel(writer, sheet_name="3D COM Points", index=False)
        readme.to_excel(writer, sheet_name="README", index=False)
    return out_path


# Tiny self-check so a broken runtime is obvious before real data.
_right, _ = chord_direction(np.array([1.0, 0.0]), np.array([0.0, 0.0]))
_up, _ = chord_direction(np.array([0.0, 1.0]), np.array([0.0, 0.0]))
_demo_angle = angle_between_lines(_right, _up)
if not np.isclose(_demo_angle, 90.0):
    raise RuntimeError(f"algorithm self-check failed: perpendicular chords should be 90, got {_demo_angle}")
print("Algorithm loaded. Default upper line T1-T2, lower line T8-L2. Perpendicular-chord self-check = 90 deg.")


## 3. If the label indices change, edit only this block

Change the numbers, not the level names. Missing T1 / T2 / T8 / L2 fails QC.



In [ ]:
#@title 3. If the label indices change, edit only this block
# Change the numbers or the level lists. Do not change the spelling of T1, T2, T8, or L2.
T1 = 6  #@param {type:"integer"}
T2 = 50  #@param {type:"integer"}
T8 = 57  #@param {type:"integer"}
L2 = 63  #@param {type:"integer"}
upper_levels = "T1,T2"  #@param {type:"string"}
lower_levels = "T8,L2"  #@param {type:"string"}

LEVEL_TO_LABEL = {"T1": int(T1), "T2": int(T2), "T8": int(T8), "L2": int(L2)}
UPPER_LEVELS = parse_level_list(upper_levels)
LOWER_LEVELS = parse_level_list(lower_levels)
for level in [*UPPER_LEVELS, *LOWER_LEVELS]:
    if level not in LEVEL_TO_LABEL:
        raise ValueError(
            f"{level} is not in the label map. Add it next to T1/T2/T8/L2, or set the lines back to T1,T2 / T8,L2."
        )
CONFIG = {
    "level_to_label": LEVEL_TO_LABEL,
    "upper_levels": UPPER_LEVELS,
    "lower_levels": LOWER_LEVELS,
}
print("Label map:", LEVEL_TO_LABEL)
print("Upper line:", "-".join(UPPER_LEVELS), "  Lower line:", "-".join(LOWER_LEVELS))
print("If any required label is missing, QC fails and no angle is forced.")


## 4. Upload segmentations



In [ ]:
#@title 4. Upload segmentations { display-mode: "form" }
data_source = "Google Drive folder"  #@param ["Google Drive folder", "Upload zip", "Upload nii files", "Local / already-mounted folder"]
folder_path = "/content/drive/MyDrive/Joey3_seg"  #@param {type:"string"}

from pathlib import Path
import shutil

UPLOAD_ROOT = Path("/content/3d_com_uploads")
CASES: list[Path] = []


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _mount_drive() -> None:
    if not _in_colab():
        print("Not in Colab; skipping Drive mount. Set folder_path to a local folder.")
        return
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


def _save_uploads(destination: Path) -> list[Path]:
    from google.colab import files
    destination.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    saved = []
    for name, payload in uploaded.items():
        path = destination / Path(name).name
        path.write_bytes(payload)
        saved.append(path)
    return saved


if data_source == "Google Drive folder":
    _mount_drive()
    CASES = discover_cases(Path(folder_path))
elif data_source == "Local / already-mounted folder":
    CASES = discover_cases(Path(folder_path))
elif data_source == "Upload zip":
    if not _in_colab():
        raise SystemExit("Zip upload works only in Colab. Use Local and paste a folder path.")
    if UPLOAD_ROOT.exists():
        shutil.rmtree(UPLOAD_ROOT)
    saved = _save_uploads(UPLOAD_ROOT)
    zips = [path for path in saved if path.suffix.lower() == ".zip"]
    if not zips:
        raise SystemExit("No zip received. Re-run this block and choose a .zip file.")
    extract_dir = UPLOAD_ROOT / "unzipped"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(extract_dir)
    CASES = discover_cases(extract_dir)
elif data_source == "Upload nii files":
    if not _in_colab():
        raise SystemExit("NIfTI upload works only in Colab. Use Local and paste a folder path.")
    if UPLOAD_ROOT.exists():
        shutil.rmtree(UPLOAD_ROOT)
    saved = _save_uploads(UPLOAD_ROOT)
    CASES = discover_cases(UPLOAD_ROOT)
    if not CASES:
        CASES = [path for path in saved if path.name.lower().endswith((".nii", ".nii.gz"))]
else:
    raise SystemExit(f"Unknown data_source: {data_source}")

if not CASES:
    raise SystemExit(
        "No segmentations found. Check that: 1) the path is correct; "
        "2) the zip is not an extra empty wrapper folder; "
        "3) files are .nii or .nii.gz; "
        "4) they look like case/seg.nii.gz or a folder of NIfTI files."
    )

print(f"Found {len(CASES)} case(s).")
for path in CASES[:20]:
    print(f"  {case_id_from_path(path)}  <-  {path}")
if len(CASES) > 20:
    print(f"  ... and {len(CASES) - 20} more")


## 5. Check labels



In [ ]:
#@title 5. Check labels in the first case { display-mode: "form" }
first = CASES[0]
print("First case:", case_id_from_path(first))
print("Path:", first)
labels_in_file = unique_labels(first)
print("Non-zero labels in the file:", labels_in_file)

print("The label map needs:")
missing = []
for level in [*UPPER_LEVELS, *LOWER_LEVELS]:
    label = LEVEL_TO_LABEL[level]
    ok = label in labels_in_file
    mark = "OK" if ok else "MISSING"
    print(f"  {level} = {label}  {mark}")
    if not ok:
        missing.append(f"{level}={label}")

if missing:
    raise SystemExit(
        "The first case does not match the label map: "
        + ", ".join(missing)
        + ". Go back to Block 3, fix the numbers, then re-run from Block 3 downward."
    )
print("All required labels are present in the first case. Continue to batch calculation.")


## 6. Batch calculate



In [ ]:
#@title 6. Batch-calculate kyphosis angles { display-mode: "form" }
from IPython.display import display

rows = []
point_rows = []
for index, seg_path in enumerate(CASES, start=1):
    measurement, points = measure_case(
        seg_path,
        LEVEL_TO_LABEL,
        UPPER_LEVELS,
        LOWER_LEVELS,
    )
    rows.append(measurement)
    point_rows.extend(points)
    status = "PASS" if measurement["qc_pass"] else "FAIL"
    angle = measurement["angle_deg"]
    angle_text = f"{angle:.3f}" if np.isfinite(angle) else "NA"
    reason = measurement["qc_reason"]
    extra = f"  {reason}" if reason else ""
    print(f"[{index:02d}/{len(CASES):02d}] {measurement['case_id']}: {status} {angle_text}{extra}")

RESULTS = pd.DataFrame(rows)
POINTS = pd.DataFrame(point_rows)
print()
print(f"Done: {int(RESULTS['qc_pass'].sum())} / {len(RESULTS)} passed.")
display(RESULTS[["case_id", "angle_deg", "qc_pass", "missing_labels", "qc_reason",
                 "upper_chord_length_mm", "lower_chord_length_mm"]])


## 7. Download Excel



In [ ]:
#@title 7. Write and download Excel { display-mode: "form" }
from pathlib import Path

output_name = "3D_COM_kyphosis_angles.xlsx"  #@param {type:"string"}
output_dir = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_XLSX = write_excel(RESULTS, POINTS, CONFIG, output_dir / output_name)
print("Wrote:", OUTPUT_XLSX.resolve())

try:
    from google.colab import files
    files.download(str(OUTPUT_XLSX))
    print("Colab should prompt a download. If not, look for the xlsx in the left Files pane.")
except ImportError:
    print("Not in Colab. Open the Excel file from the path printed above.")


## 8. Troubleshooting

- **MISSING labels.** Block 5 stops. Open Block 3, set T1/T2/T8/L2 to the numbers in the file, then re-run from Block 3 downward.
- **Folder not found.** Drive paths start at `/content/drive/MyDrive/...`. Confirm the folder is visible in the left Files pane.
- **Zip unpacks to 0 cases.** The zip often has an extra wrapper folder. Open it and confirm you see `001/seg.nii.gz` or a set of `.nii.gz` files.
- **Upload fails or files are huge.** Do not upload dozens of large NIfTI files in the browser. Use Google Drive.
- **The Colab session dropped.** Files in `/content` are gone. Use **Runtime → Run all** again.
- **You changed labels but the angles did not change.** Re-run Blocks 3 through 7. Do not re-run only the last block.
- **The angle differs from a human Cobb.** This is a vertebral centerline angle, not an endplate Cobb. Unusual morphology or segmentation error can shift it by a few degrees.



## Demo



In [ ]:
#@title Demo: smoke test without an upload (optional) { display-mode: "form" }
# Build four small spheres in memory to confirm install + algorithm. This is not a real case.

affine = np.diag([1.0, 1.0, 1.0, 1.0])
volume = np.zeros((40, 40, 80), dtype=np.int16)
centers = {
    "T1": ((20, 10, 70), LEVEL_TO_LABEL["T1"]),
    "T2": ((20, 12, 62), LEVEL_TO_LABEL["T2"]),
    "T8": ((20, 22, 28), LEVEL_TO_LABEL["T8"]),
    "L2": ((20, 18, 10), LEVEL_TO_LABEL["L2"]),
}
ii, jj, kk = np.ogrid[:40, :40, :80]
for (i, j, k), label in centers.values():
    mask = (ii - i) ** 2 + (jj - j) ** 2 + (kk - k) ** 2 <= 9
    volume[mask] = label

demo_dir = Path("/tmp/3d_com_demo")
demo_dir.mkdir(parents=True, exist_ok=True)
demo_seg = demo_dir / "seg.nii.gz"
nib.save(nib.Nifti1Image(volume, affine), str(demo_seg))
demo_row, demo_points = measure_case(demo_seg, LEVEL_TO_LABEL, UPPER_LEVELS, LOWER_LEVELS)
print("Demo case:", demo_row["case_id"], "angle =", f"{demo_row['angle_deg']:.2f} deg", "qc =", demo_row["qc_pass"])
print("This is only a smoke test. Use Blocks 4-7 with real segmentations for results.")
